In [1]:
import sqlite3

conn = sqlite3.connect(":memory:")

conn.execute("""
CREATE TABLE transactions (
id INTEGER PRIMARY KEY,
date TEXT NOT NULL,
description TEXT NOT NULL,
amount REAL NOT NULL,
category TEXT,
category_source TEXT
)
""")

In [4]:
conn.execute(
    "INSERT INTO transactions (date, description, amount) VALUES (?, ?, ?)",
    ("2026-10-29", "PRET A MANGER", -3.50)
)


In [9]:
conn.execute("SELECT * FROM transactions").fetchall()

[(1, '2026-10-29', 'PRET A MANGER', -3.5, 'Eating out', 'rule'),
 (2, '2026-10-29', 'PRET A MANGER', -3.5, 'Work expense', 'manually'),
 (3, '2026-05-01 00:00:00', 'RENT PAYMENT', -1150.0, 'Rent', None),
 (4, '2026-05-02 00:00:00', 'TESCO STORES', -19.22, 'Groceries', None),
 (5, '2026-05-04 00:00:00', 'AMAZON MKTPLACE', -44.61, 'Shopping', None),
 (6, '2026-05-08 00:00:00', 'AMAZON MKTPLACE', -52.39, 'Shopping', None),
 (7, '2026-05-09 00:00:00', 'PRET A MANGER', -5.79, 'Eating out', None),
 (8, '2026-05-10 00:00:00', 'BOOTS', -21.19, 'Self care', None),
 (9, '2026-05-12 00:00:00', 'NETFLIX.COM', -10.99, 'Subscriptions', None),
 (10, '2026-05-15 00:00:00', 'AMAZON MKTPLACE', -21.67, 'Shopping', None),
 (11, '2026-05-18 00:00:00', 'UBER *TRIP', -18.23, 'Transport', None),
 (12, '2026-05-23 00:00:00', 'SAINSBURYS', -35.92, 'Groceries', None),
 (13, '2026-05-25 00:00:00', 'SALARY ACME LTD', 1850.0, 'Income', None),
 (14, '2026-05-26 00:00:00', 'TESCO STORES', -51.26, 'Groceries', None),


In [6]:
conn.execute(
    "UPDATE transactions SET category = ?, category_source = ? WHERE id = ?",
    ("Eating out", "rule", 1)
)
conn.execute(
    "UPDATE transactions SET category = ?, category_source = ? WHERE id = ?",
    ("Work expense", "manually", 2)
)


In [7]:
conn.execute("SELECT * FROM transactions WHERE category_source = 'manually'").fetchall()

[(2, '2026-10-29', 'PRET A MANGER', -3.5, 'Work expense', 'manually')]

In [8]:
from spending_analyser.loader import load_hsbc_csv
from spending_analyser.categoriser import add_categories

df = load_hsbc_csv("../data/sample/hsbc_sample_large.csv")
df = add_categories(df)
df.to_sql("transactions", conn, if_exists="append", index=False)

conn.execute("SELECT COUNT(*) FROM transactions").fetchone()

(85,)

In [11]:
conn.execute("""
SELECT category, SUM(amount) AS total
FROM transactions
WHERE amount < 0
GROUP BY category
ORDER BY total
""").fetchall()

[('Rent', -6900.0),
 ('Groceries', -526.75),
 ('Shopping', -274.13),
 ('Eating out', -174.54),
 ('Entertainment', -144.79),
 ('Transport', -136.66),
 ('Subscriptions', -114.9),
 ('Self care', -94.74),
 ('Work expense', -3.5)]

In [12]:
import pandas as pd

df_back = pd.read_sql("SELECT * FROM transactions", conn, parse_dates=["date"])
df_back.head()

,id,date,description,amount,category,category_source
0,1,2026-10-29,PRET A MANGER,-3.50,Eating out,rule
1,2,2026-10-29,PRET A MANGER,-3.50,Work expense,manually
2,3,NaT,RENT PAYMENT,-1150.00,Rent,NaN
3,4,NaT,TESCO STORES,-19.22,Groceries,NaN
4,5,NaT,AMAZON MKTPLACE,-44.61,Shopping,NaN


In [13]:
from spending_analyser.analysis import monthly_summary

monthly_summary(df_back)

,income,spending,net
date,,,
2026-10,0.0,7.0,-7.0
